# Pigeon Showcase

Demonstrates the pigeon skeleton:
- Loading with column mapping (CSV uses lab names like `Right_Tip`, mapped to canonical `right_wingtip`)
- Full mode vs simple variant
- Bilateral/unilateral conversion with round-trip verification
- Marker pairs and centre markers

In [1]:
from morphing_birds import Animal3D, SkeletonDefinition, plot_plotly
import numpy as np

## 1. Full Mode — All Markers

In [2]:
# Full mode: 19 total markers, 14 for analysis
pigeon = Animal3D("pigeon", data="../data/mean_pigeon_shape.csv")

print(f"Total markers: {pigeon.skeleton.n_markers}")
print(f"Analysis markers: {len(pigeon.analysis_indices)}")
print(f"Display-only: {pigeon.skeleton.display_only_markers}")
print(f"\nBody sections: {list(pigeon.polygons.keys())}")

# The column mapping translates lab names to canonical names automatically
print(f"\nColumn mapping examples:")
for canonical, csv_name in list(pigeon.skeleton.column_mapping.items())[:5]:
    print(f"  {canonical} <- {csv_name}")

fig = plot_plotly(pigeon, colour='dodgerblue')
fig.update_layout(title="Pigeon — Full mode (14 analysis + 5 display-only)")
fig.show()

Total markers: 19
Analysis markers: 14
Display-only: ['centre_body_base', 'head', 'centre_backpack', 'left_tailbase', 'right_tailbase']

Body sections: ['head', 'body', 'tail', 'right_armwing', 'left_armwing', 'left_handwing', 'right_handwing']

Column mapping examples:
  head <- Head
  centre_backpack <- Body_Start
  centre_body_base <- Body_End
  left_tailtip <- Left_Tail
  right_tailtip <- Right_Tail


## 2. Simple Variant — Hawk-Compatible Marker Set

The `simple` variant excludes shoulders, elbows, and other markers to create an 8-marker set comparable to the hawk. Body sections are simplified accordingly.

In [3]:
# Simple variant — like old Pigeon3D(csv, use_simple=True)
pigeon_simple = Animal3D("pigeon", data="../data/mean_pigeon_shape.csv", variant="simple")

print(f"Simple mode:")
print(f"  Analysis markers: {len(pigeon_simple.analysis_indices)}")
print(f"  Analysis names: {pigeon_simple.analysis_marker_names}")
print(f"  Body sections: {list(pigeon_simple.polygons.keys())}")

fig = plot_plotly(pigeon_simple, colour='salmon')
fig.update_layout(title="Pigeon — Simple variant (8 analysis markers, hawk-compatible)")
fig.show()

Simple mode:
  Analysis markers: 8
  Analysis names: ['left_wingtip', 'right_wingtip', 'left_wrist', 'right_wrist', 'left_secondary', 'right_secondary', 'left_tailtip', 'right_tailtip']
  Body sections: ['head', 'body', 'tail', 'left_armwing', 'right_armwing', 'left_handwing', 'right_handwing']


## 3. Marker Pairs and Bilateral Structure

In [4]:
# The skeleton knows about left/right pairing
skel = pigeon.skeleton

print("Left/right marker pairs:")
for left, right in skel.get_marker_pairs():
    print(f"  {left:30s} <-> {right}")

print(f"\nCentre markers (no pair): {skel.get_centre_markers()}")

Left/right marker pairs:
  left_wingtip                   <-> right_wingtip
  left_wrist                     <-> right_wrist
  left_secondary                 <-> right_secondary
  left_lastsecondary_tip         <-> right_lastsecondary_tip
  left_elbow                     <-> right_elbow
  left_shoulder                  <-> right_shoulder
  left_tailtip                   <-> right_tailtip
  left_tailbase                  <-> right_tailbase

Centre markers (no pair): ['centre_body_base', 'head', 'centre_backpack']


## 4. Bilateral/Unilateral Round-Trip

The `make_unilateral` function splits bilateral data into doubled unilateral observations (left mirrored to match right). `make_bilateral` reverses it perfectly.

In [5]:
from morphing_birds.bilateral import make_unilateral, make_bilateral

# Create synthetic bilateral data
np.random.seed(42)
n_frames = 20
data = np.random.rand(n_frames, skel.n_markers, 3)

# Ensure left is negative x, right is positive x
for left, right in skel.get_marker_pairs():
    l_idx = skel.marker_index(left)
    r_idx = skel.marker_index(right)
    data[:, l_idx, 0] = -np.abs(data[:, l_idx, 0])
    data[:, r_idx, 0] = np.abs(data[:, r_idx, 0])

print(f"Original bilateral: {data.shape}")

# Convert to unilateral
uni_data, is_left, _ = make_unilateral(data, skel)
print(f"Unilateral: {uni_data.shape} (frames doubled)")
print(f"Left frames: {is_left.sum()}, Right frames: {(~is_left).sum()}")

# Convert back to bilateral
reconstructed = make_bilateral(uni_data, skel, is_left)
print(f"Reconstructed: {reconstructed.shape}")

# Verify round-trip for paired markers
max_error = 0.0
for left, right in skel.get_marker_pairs():
    l_idx = skel.marker_index(left)
    r_idx = skel.marker_index(right)
    err_l = np.abs(reconstructed[:, l_idx, :] - data[:, l_idx, :]).max()
    err_r = np.abs(reconstructed[:, r_idx, :] - data[:, r_idx, :]).max()
    max_error = max(max_error, err_l, err_r)

print(f"\nRound-trip max error (paired markers): {max_error:.2e}")
print(f"Perfectly invertible: {max_error < 1e-10}")

Original bilateral: (20, 19, 3)
Unilateral: (40, 11, 3) (frames doubled)
Left frames: 20, Right frames: 20
Reconstructed: (20, 19, 3)

Round-trip max error (paired markers): 0.00e+00
Perfectly invertible: True


## 5. PCA Marker Labels

The `analysis_marker_labels()` method generates labels for PCA heatmaps — one per marker per axis.

In [6]:
# Full mode labels
labels_full = pigeon.analysis_marker_labels()
print(f"Full mode: {len(labels_full)} labels ({len(pigeon.analysis_indices)} markers x 3 axes)")
print(f"  First 9: {labels_full[:9]}")

# Simple mode labels
labels_simple = pigeon_simple.analysis_marker_labels()
print(f"\nSimple mode: {len(labels_simple)} labels ({len(pigeon_simple.analysis_indices)} markers x 3 axes)")
print(f"  First 9: {labels_simple[:9]}")

Full mode: 42 labels (14 markers x 3 axes)
  First 9: ['left_wingtip_x', 'left_wingtip_y', 'left_wingtip_z', 'right_wingtip_x', 'right_wingtip_y', 'right_wingtip_z', 'left_wrist_x', 'left_wrist_y', 'left_wrist_z']

Simple mode: 24 labels (8 markers x 3 axes)
  First 9: ['left_wingtip_x', 'left_wingtip_y', 'left_wingtip_z', 'right_wingtip_x', 'right_wingtip_y', 'right_wingtip_z', 'left_wrist_x', 'left_wrist_y', 'left_wrist_z']


## 6. Customising the Analysis Marker Set with `exclude_markers()`

You don't need a new skeleton or YAML config to drop markers from analysis. Use `exclude_markers()` at runtime. The excluded markers stay visible in plots (frozen at their default positions) but are removed from `.markers`, `.analysis_indices`, and `.analysis_marker_labels()`.

**When to use `exclude_markers()` vs `variant`**:
- **`exclude_markers()`** — ad-hoc marker removal. Readable, flexible, done in your notebook. Use this when you want to customise the analysis set for your specific experiment.
- **`variant`** — a named preset baked into the YAML config. Use this for well-defined standard configurations that multiple people reuse (e.g. `"simple"` = hawk-comparable 8-marker set).

### Example: Remove shoulder markers

The shoulder start and trailing-edge shoulder markers can have inconsistent placement between sessions. Since the shoulder largely follows the wrist trailing edge, removing it doesn't lose much information. The body polygon still renders using the default shoulder positions.

In [7]:
# Start with the full pigeon
pigeon_no_shoulder = Animal3D("pigeon", data="../data/mean_pigeon_shape.csv")

print("=== Full marker set ===")
print(f"Analysis markers ({len(pigeon_no_shoulder.analysis_indices)}):")
print(f"  {pigeon_no_shoulder.analysis_marker_names}")

# Exclude shoulder markers from analysis
# - left/right_shoulder (Shoulder_Start) — inconsistent placement
# - left/right_lastsecondary_tip (TE_shoulder) — redundant with wrist TE
pigeon_no_shoulder.exclude_markers([
    "left_shoulder", "right_shoulder",
    "left_lastsecondary_tip", "right_lastsecondary_tip",
])

print(f"\n=== After excluding shoulders ===")
print(f"Analysis markers ({len(pigeon_no_shoulder.analysis_indices)}):")
print(f"  {pigeon_no_shoulder.analysis_marker_names}")
print(f"\nDisplay-only markers (frozen at default):")
print(f"  {[pigeon_no_shoulder.marker_name_at(i) for i in pigeon_no_shoulder.display_only_indices]}")

# The polygon still renders with the default shoulder positions
fig = plot_plotly(pigeon_no_shoulder, colour='dodgerblue')
fig.update_layout(title="Pigeon — shoulders excluded from analysis (body still visible)")
fig.show()

=== Full marker set ===
Analysis markers (14):
  ['left_wingtip', 'right_wingtip', 'left_wrist', 'right_wrist', 'left_secondary', 'right_secondary', 'left_lastsecondary_tip', 'right_lastsecondary_tip', 'left_elbow', 'right_elbow', 'left_shoulder', 'right_shoulder', 'left_tailtip', 'right_tailtip']

=== After excluding shoulders ===
Analysis markers (10):
  ['left_wingtip', 'right_wingtip', 'left_wrist', 'right_wrist', 'left_secondary', 'right_secondary', 'left_elbow', 'right_elbow', 'left_tailtip', 'right_tailtip']

Display-only markers (frozen at default):
  ['left_lastsecondary_tip', 'right_lastsecondary_tip', 'left_shoulder', 'right_shoulder', 'centre_body_base', 'head', 'centre_backpack', 'left_tailbase', 'right_tailbase']


### Wings-only: exclude shoulders AND tail

To focus purely on wing morphing and gust response, additionally exclude the tailtips. This gives a clean wing-only analysis set.

In [8]:
# Start fresh — exclude shoulders AND tail
pigeon_wings = Animal3D("pigeon", data="../data/mean_pigeon_shape.csv")

pigeon_wings.exclude_markers([
    "left_shoulder", "right_shoulder",
    "left_lastsecondary_tip", "right_lastsecondary_tip",
    "left_tailtip", "right_tailtip",
])

print(f"=== Wings-only analysis set ===")
print(f"Analysis markers ({len(pigeon_wings.analysis_indices)}):")
print(f"  {pigeon_wings.analysis_marker_names}")
print(f"\nPCA labels ({len(pigeon_wings.analysis_marker_labels())}):")
for label in pigeon_wings.analysis_marker_labels():
    print(f"  {label}")

fig = plot_plotly(pigeon_wings, colour='mediumseagreen')
fig.update_layout(title="Pigeon — wings only (no shoulders, no tail)")
fig.show()

=== Wings-only analysis set ===
Analysis markers (8):
  ['left_wingtip', 'right_wingtip', 'left_wrist', 'right_wrist', 'left_secondary', 'right_secondary', 'left_elbow', 'right_elbow']

PCA labels (24):
  left_wingtip_x
  left_wingtip_y
  left_wingtip_z
  right_wingtip_x
  right_wingtip_y
  right_wingtip_z
  left_wrist_x
  left_wrist_y
  left_wrist_z
  right_wrist_x
  right_wrist_y
  right_wrist_z
  left_secondary_x
  left_secondary_y
  left_secondary_z
  right_secondary_x
  right_secondary_y
  right_secondary_z
  left_elbow_x
  left_elbow_y
  left_elbow_z
  right_elbow_x
  right_elbow_y
  right_elbow_z


### Comparison: three configurations side by side

Use `plot_plotly_compare` to overlay the different marker sets. The shapes look the same because display-only markers still render — what changes is which markers go into analysis (PCA, etc.).

In [9]:
from morphing_birds import plot_plotly_compare

# Quick summary of all three configurations
configs = {
    "Full (14 analysis)": pigeon,
    "No shoulders (10 analysis)": pigeon_no_shoulder,
    "Wings only (8 analysis)": pigeon_wings,
}

for name, p in configs.items():
    print(f"{name}:")
    print(f"  Analysis: {p.analysis_marker_names}")
    print()

Full (14 analysis):
  Analysis: ['left_wingtip', 'right_wingtip', 'left_wrist', 'right_wrist', 'left_secondary', 'right_secondary', 'left_lastsecondary_tip', 'right_lastsecondary_tip', 'left_elbow', 'right_elbow', 'left_shoulder', 'right_shoulder', 'left_tailtip', 'right_tailtip']

No shoulders (10 analysis):
  Analysis: ['left_wingtip', 'right_wingtip', 'left_wrist', 'right_wrist', 'left_secondary', 'right_secondary', 'left_elbow', 'right_elbow', 'left_tailtip', 'right_tailtip']

Wings only (8 analysis):
  Analysis: ['left_wingtip', 'right_wingtip', 'left_wrist', 'right_wrist', 'left_secondary', 'right_secondary', 'left_elbow', 'right_elbow']



### How to use these in practice

The pattern is simple: create your `Animal3D`, exclude what you don't want, then proceed with analysis. Everything downstream (`.markers`, `.analysis_marker_labels()`, `animate_plotly`) respects the exclusion automatically.

```python
# In your analysis notebook:
pigeon = Animal3D("pigeon", data="path/to/motion_data.csv")

# Choose your analysis set
pigeon.exclude_markers([
    "left_shoulder", "right_shoulder",
    "left_lastsecondary_tip", "right_lastsecondary_tip",
    "left_tailtip", "right_tailtip",  # optional: remove to focus on wings
])

# Everything downstream uses the reduced marker set
motion_data = pigeon.load_motion_data("path/to/full_flight.csv")
analysis_data = motion_data[:, pigeon.analysis_indices, :]  # only wing markers
labels = pigeon.analysis_marker_labels()                     # for PCA heatmaps
```